# UmdTask38 – Sentiment Analysis on Social Media Posts

This notebook implements a simple sentiment analysis pipeline for the
*Twitter US Airline Sentiment* dataset. The goal is to classify each tweet
as **negative**, **neutral**, or **positive**, and to understand the class
balance and text patterns using YData-profiling.

Main steps:

1. Load the airline tweets dataset and run YData-profiling to inspect
   text length, missing values, and sentiment class balance.
2. Clean the raw tweets (lowercasing, removing URLs, mentions, hashtags,
   and non-letters).
3. Split the data into train / validation / test sets.
4. Train a TF-IDF + Logistic Regression classifier.
5. Evaluate with accuracy, macro precision / recall / F1, and a confusion
   matrix on the test set.
6. Save the trained vectorizer and model for reuse in `sentiment.API.ipynb`.


In [18]:
!pip install ydata-profiling
!pip install scikit-learn ydata-profiling

In [14]:
import os
import pandas as pd
from ydata_profiling import ProfileReport

from sentiment_utils import (
    load_data,
    preprocess_dataframe,
    split_data,
    vectorize_and_train,
    evaluate_model,
    clean_text,
    INV_LABEL_MAP,
)

## Data loading and YData-profiling

Here we load the raw tweets and create a minimal YData-profiling report
to summarize the text variable and the class distribution.


In [6]:
# === Load raw data ===
csv_path = "data/Tweets.csv"   # change this name if your file is different

df_raw = load_data(csv_path)
print(df_raw.shape)
df_raw.head()

(14640, 2)


,text,airline_sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative


In [7]:
# === YData-profiling report ===
profile = ProfileReport(
    df_raw,
    title="Twitter US Airline Sentiment – Profiling Report",
    minimal=True
)
profile.to_file("airline_sentiment_profile.html")

"Profile saved to airline_sentiment_profile.html"


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 363.55it/s]


'Profile saved to airline_sentiment_profile.html'

## Text cleaning and label encoding

We apply basic tweet cleaning (remove URLs, mentions, punctuation) and
map `airline_sentiment` to integer labels (0=negative, 1=neutral, 2=positive).

In [8]:
# === Preprocess text & labels ===
df_clean = preprocess_dataframe(df_raw)

print("Raw shape:", df_raw.shape)
print("After cleaning:", df_clean.shape)
print("\nLabel distribution:")
df_clean["label"].value_counts().sort_index().rename(index={0: "negative", 1: "neutral", 2: "positive"})


Raw shape: (14640, 2)
After cleaning: (14640, 4)

Label distribution:


label
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64

## Train / validation / test split and model training

We stratify the data into train / validation / test, then fit a TF-IDF
vectorizer (1–2 grams) and a Logistic Regression classifier with
class weights to handle imbalance.


In [9]:
# === Train / validation / test split ===
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df_clean)

print("Train size:", len(X_train))
print("Val size:", len(X_val))
print("Test size:", len(X_test))

Train size: 11712
Val size: 366
Test size: 2562


In [10]:
# === Vectorize and train classifier ===
vectorizer, model = vectorize_and_train(X_train, y_train)

vectorizer, type(model)

(TfidfVectorizer(max_features=20000, min_df=5, ngram_range=(1, 2)),
 sklearn.linear_model._logistic.LogisticRegression)

## Evaluation and confusion matrix

Here we report accuracy and macro precision / recall / F1 on validation
and test sets, and plot the confusion matrix plus a detailed per-class
report.


In [11]:
# === Evaluate model on val and test sets ===
results_val = evaluate_model(vectorizer, model, X_val, y_val)
results_test = evaluate_model(vectorizer, model, X_test, y_test)

print("Validation metrics:")
for k, v in results_val.items():
    if k != "confusion_matrix" and k != "classification_report":
        print(f"  {k}: {v:.4f}")

print("\nTest metrics:")
for k, v in results_test.items():
    if k != "confusion_matrix" and k != "classification_report":
        print(f"  {k}: {v:.4f}")


Validation metrics:
  accuracy: 0.7951
  precision_macro: 0.7533
  recall_macro: 0.7730
  f1_macro: 0.7583

Test metrics:
  accuracy: 0.7760
  precision_macro: 0.7249
  recall_macro: 0.7444
  f1_macro: 0.7312


In [12]:
import matplotlib.pyplot as plt
import numpy as np

cm = results_test["confusion_matrix"]
print("Confusion matrix (rows=true, cols=pred):")
print(cm)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm, interpolation="nearest")
ax.figure.colorbar(im, ax=ax)

classes = ["negative", "neutral", "positive"]
ax.set(
    xticks=np.arange(len(classes)),
    yticks=np.arange(len(classes)),
    xticklabels=classes,
    yticklabels=classes,
    ylabel="True label",
    xlabel="Predicted label",
    title="Confusion matrix – test set",
)

plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()

print("\nDetailed classification report (test set):")
print(results_test["classification_report"])


Confusion matrix (rows=true, cols=pred):
[[1311  235   60]
 [ 107  383   52]
 [  54   66  294]]

Detailed classification report (test set):
              precision    recall  f1-score   support

    negative       0.89      0.82      0.85      1606
     neutral       0.56      0.71      0.62       542
    positive       0.72      0.71      0.72       414

    accuracy                           0.78      2562
   macro avg       0.72      0.74      0.73      2562
weighted avg       0.79      0.78      0.78      2562



In [15]:
# === Helper for manual predictions ===
def predict_sentiment(text: str) -> str:
    clean = clean_text(text)
    vec = vectorizer.transform([clean])
    pred = model.predict(vec)[0]
    return INV_LABEL_MAP[pred]

examples = [
    "I love this airline, great service and friendly staff!",
    "This was the worst flight of my life. Delayed and rude crew.",
    "The flight was okay, nothing special but nothing terrible.",
]

for t in examples:
    print(f"{predict_sentiment(t):>8}  |  {t}")


positive  |  I love this airline, great service and friendly staff!
negative  |  This was the worst flight of my life. Delayed and rude crew.
negative  |  The flight was okay, nothing special but nothing terrible.


## Saving artifacts and manual predictions

Finally we save the TF-IDF vectorizer and model to `.joblib` files, and
test a few example sentences to check if the predictions make sense.


In [16]:
# === Save trained artifacts for reuse ===
import joblib

joblib.dump(vectorizer, "tfidf_vectorizer.joblib")
joblib.dump(model, "logreg_sentiment_model.joblib")

"Saved: tfidf_vectorizer.joblib and logreg_sentiment_model.joblib"

'Saved: tfidf_vectorizer.joblib and logreg_sentiment_model.joblib'